# VoiceFL — Phase 1: LibriSpeech Data Exploration

**Purpose:** Interactive exploration of `openslr/librispeech_asr` (train-clean-100) before running any pipeline scripts.  
This notebook is your interface to the raw dataset. Work through every section before running `data/download.py`.

**Run order:** Sections are independent and re-runnable. Complete the Exploration Checklist (Section 10) before proceeding to the pipeline.

In [1]:
# Common imports — run this cell first
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

random.seed(42)
np.random.seed(42)

print('Imports OK')

Imports OK


---
## Section 1 — Dataset Loading & Schema

We load `train.clean.100` (the 100-hour clean split of LibriSpeech) from HuggingFace.

**First run:** This downloads ~6 GB and caches locally. Subsequent runs load from cache instantly.

**Schema fields:**
- `audio` — dict with `array` (float32 numpy array at 16 kHz) and `sampling_rate` (always 16000)
- `text` — uppercase transcription string (e.g. `"THE CAT SAT ON THE MAT"`)
- `speaker_id` — integer, unique per speaker (251 speakers total in this split)
- `chapter_id` — integer, which chapter/book recording session the clip is from
- `id` — string in format `speakerid-chapterid-utteranceid` (e.g. `"19-198-0000"`)

The `speaker_id` is our partitioning key — one speaker = one FL node in our simulation.  
**Important:** speaker_id will be discarded after partitioning. It never appears in any `data/nodes/` artifact.

In [2]:
import io
import os

import soundfile as sf
from datasets import Audio, load_dataset

print('Loading LibriSpeech train.clean.100 ...')
# cache_dir='../data' points to the project's data/ dir (symlink to HF cache)
ds = load_dataset('openslr/librispeech_asr', 'clean', split='train.100', cache_dir='../data')

# datasets 4.8.4 removed the soundfile backend — torchcodec is now the only
# default decoder but requires CUDA libs we don't have. Disable auto-decode
# entirely; we decode manually with soundfile where needed.
ds = ds.cast_column('audio', Audio(decode=False))

print(f'Total clips: {len(ds)}')
print(f'Column names: {ds.column_names}')
print()
print('Features schema:')
print(ds.features)

Loading LibriSpeech train.clean.100 ...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Total clips: 28539
Column names: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id']

Features schema:
{'file': Value('string'), 'audio': Audio(sampling_rate=None, decode=False, num_channels=None, stream_index=None), 'text': Value('string'), 'speaker_id': Value('int64'), 'chapter_id': Value('int64'), 'id': Value('string')}


In [3]:
import io
import soundfile as sf

def decode_audio(audio_dict: dict) -> tuple:
    """Decode an audio dict (from Audio(decode=False)) to (array, sample_rate)."""
    if audio_dict.get("bytes"):
        arr, sr = sf.read(io.BytesIO(audio_dict["bytes"]), dtype="float32")
    else:
        arr, sr = sf.read(audio_dict["path"], dtype="float32")
    if arr.ndim > 1:
        arr = arr.mean(axis=1)  # stereo → mono
    return arr.astype(np.float32), sr

def audio_duration(audio_dict: dict) -> float:
    """Get clip duration in seconds without full decode (header only)."""
    if audio_dict.get("bytes"):
        info = sf.info(io.BytesIO(audio_dict["bytes"]))
    else:
        info = sf.info(audio_dict["path"])
    return info.frames / info.samplerate

print("Helper functions defined: decode_audio, audio_duration")

Helper functions defined: decode_audio, audio_duration


In [4]:
# Inspect one example row in full
example = ds[0]

print('=== Example row ===')
print(f"id:          {example['id']}")
print(f"speaker_id:  {example['speaker_id']}")
print(f"chapter_id:  {example['chapter_id']}")
print(f"text:        {example['text'][:80]}...")
print()

arr, sr = decode_audio(example['audio'])
print(f"audio array shape: {arr.shape}")
print(f"audio dtype:       {arr.dtype}")
print(f"sampling rate:     {sr} Hz")
print(f"duration:          {len(arr)/sr:.2f} s")
print(f"value range:       [{arr.min():.4f}, {arr.max():.4f}]")

=== Example row ===
id:          374-180298-0000
speaker_id:  374
chapter_id:  180298
text:        CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW ...

audio array shape: (232480,)
audio dtype:       float32
sampling rate:     16000 Hz
duration:          14.53 s
value range:       [-0.5280, 0.5631]


In [5]:
from tqdm.notebook import tqdm

durations = []
speaker_data = defaultdict(lambda: {'clips': 0, 'total_s': 0.0})

print('Computing statistics over all clips...')
for row in tqdm(ds, total=len(ds)):
    dur = audio_duration(row['audio'])
    sid = row['speaker_id']
    durations.append(dur)
    speaker_data[sid]['clips'] += 1
    speaker_data[sid]['total_s'] += dur

durations = np.array(durations)
total_hours = durations.sum() / 3600
clips_per_speaker = np.array([v['clips'] for v in speaker_data.values()])

print()
print('=== Dataset Statistics ===')
print(f"Total clips:         {len(durations):,}")
print(f"Total speakers:      {len(speaker_data)}")
print(f"Total audio:         {total_hours:.2f} hours")
print()
print('=== Clip Duration (seconds) ===')
print(f"Min:     {durations.min():.2f}s")
print(f"Max:     {durations.max():.2f}s")
print(f"Mean:    {durations.mean():.2f}s")
print(f"Median:  {np.median(durations):.2f}s")
print(f"p5:      {np.percentile(durations, 5):.2f}s")
print(f"p25:     {np.percentile(durations, 25):.2f}s")
print(f"p75:     {np.percentile(durations, 75):.2f}s")
print(f"p95:     {np.percentile(durations, 95):.2f}s")
print()
print('=== Clips per Speaker ===')
print(f"Min:     {clips_per_speaker.min()}")
print(f"Max:     {clips_per_speaker.max()}")
print(f"Mean:    {clips_per_speaker.mean():.1f}")

Computing statistics over all clips...


  0%|          | 0/28539 [00:00<?, ?it/s]


=== Dataset Statistics ===
Total clips:         28,539
Total speakers:      251
Total audio:         100.59 hours

=== Clip Duration (seconds) ===
Min:     1.41s
Max:     24.52s
Mean:    12.69s
Median:  13.99s
p5:      4.25s
p25:     11.62s
p75:     15.16s
p95:     16.09s

=== Clips per Speaker ===
Min:     26
Max:     166
Mean:    113.7


---
## Section 2 — Basic Statistics

Compute dataset-wide statistics. This gives us the full picture before zooming into speaker-level analysis.

**Note:** Iterating the full dataset takes ~2–3 minutes. Results are printed as a summary table.

In [6]:
# Per-speaker detailed stats
speaker_clips = defaultdict(list)  # speaker_id -> list of durations

for row in tqdm(ds, total=len(ds), desc='Grouping by speaker'):
    dur = audio_duration(row['audio'])
    speaker_clips[row['speaker_id']].append(dur)

speaker_stats = []
for sid, durs in speaker_clips.items():
    durs_arr = np.array(durs)
    speaker_stats.append({
        'speaker_id': sid,
        'clip_count': len(durs_arr),
        'total_duration_s': durs_arr.sum(),
        'mean_duration_s': durs_arr.mean(),
        'std_duration_s': durs_arr.std(),
    })

speaker_stats.sort(key=lambda x: x['clip_count'], reverse=True)

print('=== Top 10 Speakers by Clip Count ===')
print(f"{'Speaker':>10} {'Clips':>7} {'Total(min)':>12} {'Mean(s)':>9} {'Std(s)':>8}")
print('-' * 52)
for s in speaker_stats[:10]:
    print(f"{s['speaker_id']:>10} {s['clip_count']:>7} {s['total_duration_s']/60:>12.1f} {s['mean_duration_s']:>9.2f} {s['std_duration_s']:>8.2f}")

print()
print('=== Bottom 10 Speakers by Clip Count ===')
print(f"{'Speaker':>10} {'Clips':>7} {'Total(min)':>12} {'Mean(s)':>9} {'Std(s)':>8}")
print('-' * 52)
for s in speaker_stats[-10:]:
    print(f"{s['speaker_id']:>10} {s['clip_count']:>7} {s['total_duration_s']/60:>12.1f} {s['mean_duration_s']:>9.2f} {s['std_duration_s']:>8.2f}")

print(f"\nSpeakers with < 50 clips: {sum(1 for s in speaker_stats if s['clip_count'] < 50)}")

Grouping by speaker:   0%|          | 0/28539 [00:00<?, ?it/s]

=== Top 10 Speakers by Clip Count ===
   Speaker   Clips   Total(min)   Mean(s)   Std(s)
----------------------------------------------------
       211     166         25.2      9.12     4.71
      4014     165         25.2      9.16     4.67
       730     161         25.2      9.39     4.52
      2989     155         25.2      9.76     4.73
      8063     155         25.1      9.71     4.38
      4195     140         25.1     10.75     4.36
        27     138         20.1      8.76     4.87
       125     138         25.2     10.95     4.35
       118     137         25.1     10.97     4.59
      1867     137         25.2     11.02     4.37

=== Bottom 10 Speakers by Clip Count ===
   Speaker   Clips   Total(min)   Mean(s)   Std(s)
----------------------------------------------------
       289      86         19.6     13.71     2.66
      6925      86         17.1     11.91     4.13
       458      82         17.3     12.66     4.07
      4214      81         17.4     12.92     2.8

---
## Section 3 — Speaker-Level Analysis

Understanding the per-speaker distribution is critical for FL node design.  
High variance in clip counts = natural non-IID heterogeneity — exactly what makes FL interesting.

We also compute `std_duration_s` per speaker, which is our heterogeneity proxy used in `download.py`  
for stratified speaker selection.

---
## Section 4 — Audio Quality Checks

LibriSpeech is a carefully curated dataset, so we expect it to be very clean.  
This section confirms that — or surfaces any surprises before the pipeline runs.

We sample 50 clips (5 per speaker from 10 random speakers) and run 4 checks:

1. **Silence** — `abs(audio) < 0.01` fraction > 0.5 → mostly silent clip
2. **Clipping** — `abs(audio) > 0.99` fraction > 0.01 → amplitude clipping
3. **Duration outliers** — shorter than 1s or longer than 30s
4. **Sample rate** — must be exactly 16000 Hz

In [7]:
from collections import Counter

sample_200_idx = random.sample(range(len(ds)), 200)
sample_200 = [ds[i] for i in sample_200_idx]

# Show 10 example (duration, transcription) pairs
print('=== 10 Example (duration, transcription) pairs ===')
for row in sample_200[:10]:
    dur = audio_duration(row['audio'])
    print(f"  [{dur:.1f}s] {row['text'][:90]}")

# Vocabulary stats
all_words = []
word_counts_per_clip = []
empty_count = 0
for row in sample_200:
    words = row['text'].split()
    all_words.extend(words)
    word_counts_per_clip.append(len(words))
    if len(words) == 0:
        empty_count += 1

word_freq = Counter(all_words)
unique_words = len(set(all_words))
total_words = len(all_words)

print()
print('=== Vocabulary Stats (200-clip sample) ===')
print(f"Total words:      {total_words}")
print(f"Unique words:     {unique_words}")
print(f"Richness ratio:   {unique_words/total_words:.3f}")
print(f"Empty transcriptions: {empty_count}")
print()
print('=== 10 Most Common Words ===')
for word, count in word_freq.most_common(10):
    print(f"  {word:<20} {count}")
print()
print('=== 10 Least Common Words (appearing once) ===')
rare = [w for w, c in word_freq.items() if c == 1]
for w in list(rare)[:10]:
    print(f"  {w}")

=== 10 Example (duration, transcription) pairs ===
  [12.2s] THAT WAS WHAT MADE ME MISERABLE THEN YOU MUST OF COURSE MAKE ALL POSSIBLE REPARATION ANSWE
  [13.7s] CARSON HAD ENTERED THE VALLEY ALONG THE SOUTHERN SIDE OF THE BAY BUT THE COUNTRY THEN WAS 
  [14.8s] WOULD NOT SHARE HIM WITH THE TOWN WHEN CAROL SAW THEM VIDA WAS HAZY ABOUT EVERYTHING EXCEP
  [13.6s] THERE ARE TWO POEMS BY GEORGE STERLING THAT I HAVE HAD IN MIND FOR MANY A DAY AS CONCEPTIO
  [11.8s] WHAT I CAN'T GO OUT OTHERWISE THAN MASKED HERE I'M CONCEALED NO ONE KNOWS THAT I'M HERE BU
  [4.1s] WHAT DO YOU KNOW ASKED OLD MOTHER NATURE
  [15.2s] TURNING AGAIN AND AGAIN TO THE WINDOW LISTENING TO THE SOFT RUSH OF THE TRAINS THE FAINT H
  [13.6s] AND LOOKING WITH SAVAGE DISTRUST AT EVERYONE MY SCHOOLFELLOWS MET ME WITH SPITEFUL AND MER
  [4.5s] AFTER THE ASSEMBLY WAS BROKEN UP RECEIVED HIM INTO HIS HOUSE
  [2.2s] BUT THE MAGPIE IN THE TREE WHO LIKE

=== Vocabulary Stats (200-clip sample) ===
Total words:      6838
Unique wor

---
## Section 5 — Transcription Analysis

We look at the text side of the dataset. LibriSpeech transcriptions are read speech from public-domain books  
— expect clean, formal language with no abbreviations or slang.

This is also useful context for WER evaluation: the vocabulary is rich but not colloquial.

In [8]:
import IPython.display as ipd

# Random clip
idx = random.randint(0, len(ds) - 1)
sample = ds[idx]
arr, sr = decode_audio(sample['audio'])
print('=== Random clip ===')
print(f"Speaker: {sample['speaker_id']}")
print(f"Text:    {sample['text'][:80]}")
print(f"Duration: {len(arr)/sr:.2f}s")
ipd.display(ipd.Audio(arr, rate=sr))

=== Random clip ===
Speaker: 4267
Text:    SHE CRIED PUTTING HER HAND OVER MY LIPS AND GETTING IT WELL KISSED IN CONSEQUENC
Duration: 5.35s


In [9]:
# Longest clip in dataset
longest_idx = max(all_lens, key=lambda x: x[0])[1]
sample = ds[longest_idx]
arr, sr = decode_audio(sample['audio'])
print('=== Longest clip ===')
print(f"Duration: {len(arr)/sr:.2f}s")
print(f"Text:    {sample['text'][:80]}...")
ipd.display(ipd.Audio(arr, rate=sr))

NameError: name 'all_lens' is not defined

In [ ]:
import librosa

audio, _ = decode_audio(ds[0]['audio'])

# Normalize to [-1, 1]
audio = audio / (np.max(np.abs(audio)) + 1e-8)

# Extract log-mel filterbank
mel = librosa.feature.melspectrogram(
    y=audio, sr=16000, n_mels=80, n_fft=400,
    hop_length=160, fmin=0.0, fmax=8000.0
)
log_mel = librosa.power_to_db(mel, ref=np.max).T  # shape: (T, 80)

print(f'Audio duration:   {len(audio)/16000:.2f}s  ({len(audio)} samples)')
print(f'Feature shape:    {log_mel.shape}  (T={log_mel.shape[0]} frames, 80 mel bins)')
print(f'Frames / second:  {log_mel.shape[0] / (len(audio)/16000):.1f}')
print(f'Value range:      [{log_mel.min():.1f}, {log_mel.max():.1f}] dB')
print(f'dtype:            {log_mel.dtype}')
print()
print('Shape relationship check:')
expected_T = int(np.ceil(len(audio) / 160))
match = abs(expected_T - log_mel.shape[0]) <= 2
print(f'  ceil({len(audio)} / 160) = {expected_T}  →  actual T = {log_mel.shape[0]}  {"✓" if match else "MISMATCH"}')

---
## Section 7 — Feature Extraction Preview

This section shows exactly what `data/features.py` will produce — manually, on one clip.

**Parameters (fixed for the whole project):**
- `n_mels = 80` — 80 mel frequency bins
- `n_fft = 400` — 25ms window at 16 kHz (400 samples)
- `hop_length = 160` — 10ms hop at 16 kHz (160 samples)
- `f_min = 0.0`, `f_max = 8000.0`

**Output shape:** `(T, 80)` where `T = ceil(num_samples / hop_length)`.  
A 1-second clip → ~100 frames. A 5-second clip → ~500 frames.

Verify that the shape and value range look correct before running `features.py`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6),
                         gridspec_kw={'height_ratios': [1, 3]})

# Waveform
times = np.linspace(0, len(audio)/16000, len(audio))
axes[0].plot(times, audio, linewidth=0.3, color='steelblue')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Waveform')
axes[0].set_xlim([0, len(audio)/16000])

# Mel spectrogram
im = axes[1].imshow(log_mel.T, aspect='auto', origin='lower',
                    cmap='magma', interpolation='nearest')
plt.colorbar(im, ax=axes[1], label='Log Mel Energy (dB)')
axes[1].set_xlabel('Time frames')
axes[1].set_ylabel('Mel frequency bin')
axes[1].set_title(f'Log-Mel Spectrogram  shape={log_mel.shape}')

plt.tight_layout()
plt.show()

---
## Section 8 — Speaker Selection Preview

We need to choose 20 speakers from 251 for our FL nodes.  
Our strategy: **stratified sampling by `duration_std`** — the standard deviation of clip durations per speaker.

This ensures we pick speakers from all parts of the diversity spectrum:  
very consistent speakers AND highly variable speakers — mirroring real-world FL user diversity.

This cell previews that selection so you can sanity-check it before it gets locked in by `download.py`.

In [ ]:
# Add duration_std to speaker_stats
for s in speaker_stats:
    s['duration_std'] = speaker_clips[s['speaker_id']]

# Recompute properly
for s in speaker_stats:
    durs = np.array(speaker_clips[s['speaker_id']])
    s['duration_std'] = float(durs.std())

# Filter: must have >= 50 clips
eligible = [s for s in speaker_stats if s['clip_count'] >= 50]
print(f'Eligible speakers (≥50 clips): {len(eligible)} / {len(speaker_stats)}')

# Rank by duration_std
eligible_sorted = sorted(eligible, key=lambda x: x['duration_std'])
for rank, s in enumerate(eligible_sorted):
    s['diversity_rank'] = rank + 1

# Stratified selection: 20 equal-rank buckets, pick median from each
n_select = 20
n_eligible = len(eligible_sorted)
bucket_size = n_eligible / n_select

selected_preview = []
for i in range(n_select):
    start = int(i * bucket_size)
    end   = int((i + 1) * bucket_size)
    bucket = eligible_sorted[start:end]
    mid = bucket[len(bucket) // 2]
    selected_preview.append(mid)

print(f'\nStratified selection preview: {len(selected_preview)} speakers')
print(f"{'Speaker':>10} {'Clips':>7} {'Diversity rank':>16} {'Duration std(s)':>16}")
print('-' * 55)
for s in selected_preview:
    print(f"{s['speaker_id']:>10} {s['clip_count']:>7} {s['diversity_rank']:>16} {s['duration_std']:>16.3f}")

---
## Section 9 — Cleaning Decision Cell

This is the most important cell. Edit the `CLEANING_CONFIG` dict below, then run the cell.  
The config is written to `data/cleaning_config.json` and applied automatically by `pii_masking.py` and `features.py`.

**Defaults are conservative.** LibriSpeech is very clean, so you probably don't need to change anything.  
But the decision should be yours, not the pipeline's.

In [ ]:
import os, json

CLEANING_CONFIG = {
    # Filter clips shorter than this (seconds). Set to 0.0 to disable.
    'min_duration_s': 1.0,

    # Filter clips longer than this. Set to null/None to disable.
    'max_duration_s': 30.0,

    # Drop clips where silence fraction exceeds threshold. 0.0 = disable.
    'max_silence_fraction': 0.5,

    # Normalize audio to [-1, 1] before feature extraction. Recommended: True.
    'normalize_audio': True,

    # If True, truncate long clips instead of dropping them.
    'truncate_long_clips': False,
}

os.makedirs('data', exist_ok=True)
with open('data/cleaning_config.json', 'w') as f:
    json.dump(CLEANING_CONFIG, f, indent=2)

print('Cleaning config saved to data/cleaning_config.json')
print()
for k, v in CLEANING_CONFIG.items():
    print(f'  {k}: {v}')

---
## Section 10 — Exploration Checklist

Before running the pipeline, confirm every item below.

```
Pre-Pipeline Checklist
────────────────────────────────────────────────────────────
 [ ] Dataset loaded correctly (28,539 clips, 251 speakers)
 [ ] Section 2: Audio quality checks reviewed
       → Issues noted or confirmed clean
 [ ] Section 6: Listened to at least 3 audio samples
 [ ] Section 7: Feature extraction preview shape looks correct
       → Shape is (T, 80), dtype float32, range ~[-80, 0] dB
 [ ] Section 8: Speaker selection preview looks reasonable
       → 20 speakers spread across diversity range
 [ ] Section 9: Cleaning config saved to data/cleaning_config.json
────────────────────────────────────────────────────────────
 Ready to run: python data/download.py
```

### Running order after this notebook:

```bash
# From project root (voicefl/)
python data/download.py        # S1: analyze speakers, select 20, save speaker_selection.json
python data/pii_masking.py     # S2a: strip PII, save raw_clips.pkl per node
python data/features.py        # S2b: extract log-mel features, DELETE raw_clips.pkl
python data/partition.py       # S2c: validate partition, save partition_manifest.json
python data/generate_report.py # Generate figures and documentation report
```